# Hyperliquid trade analytics

Explore ATR-based stop distance, adverse funding reserve, and historical five-day expected PnL for one perp position. This notebook is read-only: it uses public market data and never signs or submits orders. The reusable calculations live in `hyperliquid_trade_analytics.py`.

## 1. Configure the position

Use the Hyperliquid market name. For HIP-3 markets, set `DEX` to the market namespace and `COIN` to the unqualified ticker (or provide the qualified name).

In [ ]:
COIN = "BTC"
DEX = ""
SIDE = "LONG"  # LONG or SHORT
QUANTITY = 1.0
HORIZON_DAYS = 5
ATR_PERIOD = 14
STOP_ATR_MULTIPLE = 1.25
FUNDING_STRESS_PERCENTILE = 0.95

## 2. Load the shared analytics functions

In [ ]:
import pandas as pd

from hyperliquid_trade_analytics import (
    adverse_funding_cost_per_unit,
    expected_position_pnl,
    fetch_daily_candles,
    fetch_funding_history,
    fetch_mark_and_oracle,
    side_to_direction,
    wilder_atr,
)

if ":" in COIN:
    prefix, COIN = COIN.split(":", 1)
    DEX = DEX or prefix
MARKET = f"{DEX}:{COIN}" if DEX else COIN
MARKET

## 3. Fetch completed daily candles and hourly funding

The helper excludes the current incomplete daily candle. Funding-history retrieval paginates API responses so estimates use the requested interval rather than only the first capped batch.

In [ ]:
candles = fetch_daily_candles(COIN, DEX, lookback_days=365)
funding = fetch_funding_history(COIN, DEX, lookback_days=20)
display(candles.tail())
display(funding.tail())

## 4. Calculate ATR stop and funding reserve

Adverse funding is the selected percentile of historical direction-adjusted rolling funding over the holding horizon, valued at the current oracle price. It is a reserve estimate, not a guarantee against future rates or execution gaps.

In [ ]:
direction = side_to_direction(SIDE)
atr = wilder_atr(candles, ATR_PERIOD)
mark_price, oracle_price = fetch_mark_and_oracle(COIN, DEX)
stop_distance = atr * STOP_ATR_MULTIPLE
stop_price = mark_price - direction * stop_distance
target_price = mark_price + direction * stop_distance
funding_reserve_per_unit = adverse_funding_cost_per_unit(
    funding,
    SIDE,
    oracle_price,
    horizon_hours=HORIZON_DAYS * 24,
    percentile=FUNDING_STRESS_PERCENTILE,
)
pd.Series({
    "Market": MARKET,
    "Side": SIDE.upper(),
    "Mark used for illustration": mark_price,
    f"ATR({ATR_PERIOD})": atr,
    "Stop distance": stop_distance,
    "Stop price": stop_price,
    "1:1 target price": target_price,
    f"Adverse {HORIZON_DAYS}d funding reserve / unit": funding_reserve_per_unit,
})

## 5. Estimate historical expected PnL

The estimate is the mean signed historical forward price return over the selected number of completed daily bars, plus separately estimated funding PnL. It is not a forecast guarantee and does not include fees or slippage.

In [ ]:
expected = expected_position_pnl(
    candles,
    funding,
    SIDE,
    quantity=QUANTITY,
    mark_price=mark_price,
    oracle_price=oracle_price,
    horizon_days=HORIZON_DAYS,
)
pd.Series({
    "Expected price PnL": expected.price_pnl,
    "Expected funding PnL": expected.funding_pnl,
    "Expected PnL including funding": expected.net_pnl,
    "Mean historical forward return": expected.mean_forward_return,
    "Completed forward-return observations": expected.forward_observations,
    "Funding windows": expected.funding_observations,
})

## 6. Risk-sizing scratchpad

This cell illustrates the configured per-position budget: 5% of the 80% usable-equity amount, or 4% of perp equity, before the portfolio-wide 20% risk cap and available-margin check. Review current aggregate open risk before any live order; the separate executor performs those checks and remains dry-run by default.

In [ ]:
PERP_EQUITY = None  # Enter the latest combined perp equity; no wallet secret is needed.
RISK_CAP_PCT = 0.20
if PERP_EQUITY is None or PERP_EQUITY <= 0:
    raise ValueError("Set PERP_EQUITY to the latest combined perp equity before using this sizing illustration.")
POSITION_RISK_BUDGET = PERP_EQUITY * 0.80 * 0.05
TOTAL_RISK_CAP = PERP_EQUITY * RISK_CAP_PCT
RISK_PER_UNIT = stop_distance + funding_reserve_per_unit
MAX_QUANTITY_BEFORE_PORTFOLIO_CAP = POSITION_RISK_BUDGET / RISK_PER_UNIT
pd.Series({
    "Per-position risk budget": POSITION_RISK_BUDGET,
    "Portfolio open-risk cap": TOTAL_RISK_CAP,
    "Price stop + funding risk per unit": RISK_PER_UNIT,
    "Maximum quantity before aggregate-risk/margin checks": MAX_QUANTITY_BEFORE_PORTFOLIO_CAP,
})